# Agent 3 — One Question Becomes Five Searches

Decomposition: the model turns a research question into 4–6 **angles**,
each a separate search aimed at a different part of the answer. Then we
*measure* what each angle set reaches — coverage is a number, not a vibe.

**No class API key?** The precomputed angles below are a real-shaped
decomposition; everything downstream runs offline.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## The model writes the angles

In [ ]:
RESEARCH_QUESTION = "Is the Riverside Community Garden growing?"

DECOMPOSE_PROMPT = f"""Turn this research question into 4-6 search angles.
Each angle is one search aimed at a different part of the answer.
Return a JSON list of short search-query strings.
Question: {RESEARCH_QUESTION}"""

PRECOMPUTED_ANGLES = [
    "history of plot numbers at the Riverside garden",
    "Riverside garden member families census",
    "city grant funding for the Riverside garden expansion",
    "waitlist to join the Riverside garden",
]

angles = get_json(DECOMPOSE_PROMPT) if HAVE_KEY else PRECOMPUTED_ANGLES
print("Angles:")
for a in angles:
    print("  -", a)

## Measuring coverage

The mini-web holds six facts that bear on the question (we know, because we
wrote it — with your real capstone question you won't have this answer key,
which is exactly why coverage discipline matters). Each fact lives on
certain pages; an angle "reaches" a fact if its search surfaces one of
those pages.

In [ ]:
FACTS = {
    "plots grew 48 -> 60":        {"riverside-garden.org/history", "cityparks.gov/report-2026",
                                   "lakeview-news.com/garden-expands", "riverside-garden.org/about"},
    "families grew 31 -> 48":     {"lakeview-news.com/roundup-2023", "cityparks.gov/report-2026",
                                   "riverside-garden.org/about"},
    "$15,000 grant in 2025":      {"cityparks.gov/grants-2025"},
    "waitlist of 22 families":    {"riverside-garden.org/join"},
    "news covered the expansion": {"lakeview-news.com/garden-expands"},
    "lease runs to 2028":         {"cityparks.gov/grants-2025"},
}

def coverage(angle_list):
    reached_pages = set()
    for angle in angle_list:
        for url, title in search(angle):
            reached_pages.add(url)
    hit = {fact for fact, pages in FACTS.items() if pages & reached_pages}
    return hit, reached_pages

hit, pages = coverage(angles)
print(f"{len(angles)} angles reached {len(pages)} pages and {len(hit)} of {len(FACTS)} facts:")
for fact in FACTS:
    print(("  FOUND " if fact in hit else "  miss  "), fact)

## One angle alone

Run the same measurement on just the first angle. Coverage is a team
property — watch it collapse.

In [ ]:
hit1, pages1 = coverage(angles[:1])
print(f"1 angle: {len(hit1)} of {len(FACTS)} facts")
hit_all, _ = coverage(angles)
print(f"{len(angles)} angles: {len(hit_all)} of {len(FACTS)} facts")
assert len(hit1) < len(hit_all), "one angle should reach less than the full set"

## Try it

1. Write a fifth angle by hand that reaches a fact the model's angles
   missed (if any). One good angle-word is worth a lot — check what
   `search()` actually matches on.
2. Add three junk angles ("garden garden garden"). Does coverage improve?
   What did the extra angles cost in a world where each search is a paid
   call?
3. **Build turn-in:** two real questions decomposed by hand AND by the
   model, with your comparison sentences.